# IsoFLOP scaling sweep on a Colab GPU

Trains byte-level GPTs at 5 compute budgets (1e12–1e14 FLOPs) with 6 model sizes per budget, then fits:

- the IsoFLOP minima N*(C) and D*(C) (Chinchilla approach 2);
- a parametric L(N, D) = E + A/N^α + B/D^β (approach 3).

The corpus is the Python standard library that ships with the runtime, so no download is needed. Its fingerprint (Python version, size, crc32) is stored in every run record.

**Runtime → Change runtime type → T4 GPU**, then run all. Results go to Google Drive, and the sweep **resumes** after a disconnect. Download `scaling_results.zip` at the end.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/scaling-laws-from-scratch/results'
!mkdir -p {OUT}

In [ ]:
!git clone -q https://github.com/kishorhange111/scaling-laws-from-scratch.git /content/repo 2>/dev/null || git -C /content/repo pull -q
%cd /content/repo
!pip install -q -e .

In [ ]:
# The plan: which (budget, model size, tokens) runs will be trained.
!python -m scaling.cli plan --budgets 1e12 3e12 1e13 3e13 1e14 --per-budget 6 --max-tokens 3e8

## Sweep

Four processes share the GPU, and the biggest budgets go first. These models are tiny (3k–2.6M non-embedding parameters), so the GPU is mostly kernel-launch-bound. Expect tens of minutes on a T4 (an estimate, not measured). If the session drops, just rerun this cell.

In [ ]:
!python -m scaling.cli run --budgets 1e12 3e12 1e13 3e13 1e14 --per-budget 6 --max-tokens 3e8 --workers 4 --threads 1 --device cuda --out {OUT}

In [ ]:
!python -m scaling.cli fit --out {OUT}
from IPython.display import Image, Markdown, display
display(Markdown(open(f'{OUT}/fits.md').read()))
display(Image(f'{OUT}/scaling.png'))

In [ ]:
!cd {OUT} && zip -qr /content/scaling_results.zip . && ls -la /content/scaling_results.zip
from google.colab import files
files.download('/content/scaling_results.zip')